## BookMatch ETL Pipeline

### Step 1: Extract Ratings Dataset

In [2]:
import pandas as pd
df=pd.read_csv('Ratings.csv', encoding="utf-8")
df.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


### Step 2: Inspect Raw Data

In [5]:
print(df.shape)
print(df.columns)
print(df.isnull().sum())

(1149780, 3)
Index(['User-ID', 'ISBN', 'Book-Rating'], dtype='str')
User-ID        0
ISBN           0
Book-Rating    0
dtype: int64


### Step 3: Remove Implicit Ratings

In [7]:
df=df[df['Book-Rating']!=0]
print(df.shape)

(433671, 3)


### Step 4: Handle Missing Values

In [9]:
df=df.dropna()
print(df.isnull().sum())

User-ID        0
ISBN           0
Book-Rating    0
dtype: int64


### Step 5: Standardize Column Names

In [10]:
df.columns=['User-ID', 'ISBN', 'Rating']
print(df.columns)

Index(['User-ID', 'ISBN', 'Rating'], dtype='str')


### Step 6: Export Clean Ratings Dataset

In [11]:
df.to_csv('Ratings_clean.csv', index=False)
print("Done! Ratings_clean.csv saved successfully 🎉")

Done! Ratings_clean.csv saved successfully 🎉


### Step 7: Connect to MySQL

In [ ]:
import pandas as pd
from sqlalchemy import create_engine,text
import warnings
warnings.filterwarnings('ignore')

engine=create_engine(
    'mysql+pymysql://<username>:<password>/bookmatch',
     connect_args={'charset':'utf8mb4'}
)

with engine.connect() as conn:
    result = conn.execute(text("SELECT 'connected'"))
    print(result.fetchone()[0])

connected


### Step 8: Load Cleaned Datasets

In [8]:
# Load Books
book_df = pd.read_excel('Book_clean.xlsm', engine='openpyxl', sheet_name=0)

# Load Users
users_df = pd.read_excel('User_clean.xlsm', engine='openpyxl', sheet_name=0)

# Load Ratings
ratings_df = pd.read_csv('Ratings_clean.csv', encoding='utf-8-sig')

print("✅ Books:", book_df.shape)
print("✅ Users:", users_df.shape)
print("✅ Ratings:", ratings_df.shape)

✅ Books: (271030, 5)
✅ Users: (277611, 3)
✅ Ratings: (433671, 3)


### Step 9: Standardize Schema Before Loading

In [9]:
for df in[book_df, users_df, ratings_df]:
    df.columns =(df.columns
                 .str.strip()
                 .str.replace(' ','_')
                 .str.replace('-','_')
                 .str.replace('[^A-Za-z0-9_]','',regex=True))
    
    print("Books col:", book_df.columns.tolist())
    print("Users col:", users_df.columns.tolist())
    print("Ratings col:", ratings_df.columns.tolist())


Books col: ['ISBN', 'Title', 'Author', 'YOP', 'Publisher']
Users col: ['User-ID', 'Location', 'Age']
Ratings col: ['User-ID', 'ISBN', 'Rating']
Books col: ['ISBN', 'Title', 'Author', 'YOP', 'Publisher']
Users col: ['User_ID', 'Location', 'Age']
Ratings col: ['User-ID', 'ISBN', 'Rating']
Books col: ['ISBN', 'Title', 'Author', 'YOP', 'Publisher']
Users col: ['User_ID', 'Location', 'Age']
Ratings col: ['User_ID', 'ISBN', 'Rating']


### Step 10: Load Data into MySQL

In [ ]:
print("Importing Books...")
book_df.to_sql('Books', con=engine,
               if_exists='replace',
               index=False,chunksize=500,schema='bookmatch')
print(f"Books done! {len(book_df)}rows")

print("Importing Users...")
users_df.to_sql('Users',con=engine,
               if_exists='replace',
               index=False,chunksize=500,schema='bookmatch')
print(f"Users done!{len(users_df)}rows")

print("Importing Ratings...")
ratings_df.to_sql('Ratings',con=engine,
                 if_exists='replace',
                 index=False,chunksize=1000,schema='bookmatch')
print(f"Ratings done! {len(ratings_df)} rows")
      


Importing Books...
Books done!271030rows
Importing Users...
Users done!277611rows
Importing Ratings...
Ratings done! 433671 rows


### Step 11: Validate Imported Records

In [25]:
result =pd.read_sql("""
                    SELECT 'Books' AS Table_Name,COUNT(*) AS Total_Rows FROM Books
                    UNION ALL
                    SELECT 'Users', COUNT(*) FROM Users
                    UNION ALL
                    SELECT 'Ratings', COUNT(*) FROM Ratings
                    """,engine)
print(result)

  Table_Name  Total_Rows
0      Books      271030
1      Users      277611
2    Ratings      433671
